To make this model work, we need to do the following: 
* For all crashes in our study area (Oakland + Berkeley), we need to mark whether a crash is associated with a traffic island and signal
* We need to determine which other explanatory variables to include in the model:
    * Traffic signal presence (OSM)
    * Traffic refuge island presence (OSM)
    * Number of lanes (TBD)
        * Looks like you have to play with the overpass API to get this data from OSM https://stackoverflow.com/questions/56558717/query-all-roads-with-overpass-api-and-export-as-polygon
    * Ped characteristics (gender, age, etc.) (retrieve from SWITRS)
        * **How do we do this if ped characteristics are tied to crash party but we want to look at crashes overall?**
    * AADT (From replica)
    * Functional classification of road as a proxy for speed
    * Day of the week (SWITRS)
    * Lighting at intersection (TBD, possibly on OSM)

In [166]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel

In [167]:
# Load dataset
file_path = "crash_data_for_model.csv"  # Update with the actual file path
df = pd.read_csv(file_path, dtype=str)

In [168]:
# Remove irrelevant columns

df = df[[
        'COLLISION_SEVERITY',
    'AT_FAULT', # No should be reference
    'PARTY_SEX', # Male should be reference
    'PARTY_AGE',
    'RACE', # White should be reference
    'LIGHTING', # Daylight should be reference
        'DAY_OF_WEEK',
    'WEATHER_1', # Clear should be reference
    'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    'PARTY_NUMBER_KILLED',
    'PARTY_NUMBER_INJURED',
    'road_class',
    'island_id', # Convert to 1/0
    'volume', 
    'signal_id' # Convert to 1/0
]]

In [169]:
def remap_severity(severity_series):
    # Convert severity values to numeric, coercing errors to NaN
    severity_numeric = pd.to_numeric(severity_series, errors="coerce")
    # Define the mapping from original to reversed scale
    severity_mapping = {1: 4, 2: 3, 3: 2, 4: 1}
    remapped = severity_numeric.map(severity_mapping)
    # Define the order for the categorical variable
    severity_order = [1, 2, 3, 4]
    return pd.Categorical(remapped, categories=severity_order, ordered=True)

# Apply severity remapping using the dedicated function
df["COLLISION_SEVERITY"] = remap_severity(df["COLLISION_SEVERITY"])

In [170]:
# # Collapse DAY_OF_WEEK into "Weekday" (Mon-Fri) vs. "Weekend" (Sat-Sun)
# df["DAY_OF_WEEK"] = df["DAY_OF_WEEK"].apply(lambda x: "Weekday" if x in ["1", "2", "3", "4", "5"] else "Weekend")

In [171]:
# Remap at fault category
fault_dict = {'N':False, 'Y':True}
df['AT_FAULT'] = df['AT_FAULT'].map(fault_dict)
# df['AT_FAULT'] = df['AT_FAULT'].astype(int)

In [172]:
# Remap island data
print(df['island_id'].value_counts().sum())
df['island_id'] = df['island_id'].notna()#.astype(int)
print(df['island_id'].value_counts())


36
island_id
False    634
True      36
Name: count, dtype: int64


In [173]:
# Remap signal data
print(df['signal_id'].value_counts().sum())
df['signal_id'] = df['signal_id'].notna()#.astype(int)
print(df['signal_id'].value_counts())

169
signal_id
False    501
True     169
Name: count, dtype: int64


In [174]:
# Apply dummy encoding, keeping "Weekday" as the reference category
df = pd.get_dummies(df, columns=[
    'PARTY_SEX', # Male should be reference
    'RACE', # White should be reference
    'LIGHTING', # Daylight should be reference
    'WEATHER_1', # Clear should be reference
    'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    ], drop_first=True)

display(df)

,COLLISION_SEVERITY,AT_FAULT,PARTY_AGE,DAY_OF_WEEK,PARTY_NUMBER_KILLED,PARTY_NUMBER_INJURED,road_class,island_id,volume,signal_id,...,PRIMARY_COLL_FACTOR_A,PRIMARY_COLL_FACTOR_B,PRIMARY_COLL_FACTOR_C,PRIMARY_COLL_FACTOR_D,PED_ACTION_B,PED_ACTION_C,PED_ACTION_D,PED_ACTION_E,PED_ACTION_F,PED_ACTION_G
0,1,False,60,2,0,1,4,False,188606.62996098422,False,...,True,False,False,False,True,False,False,False,False,False
1,1,False,57,1,0,1,5,False,1183902.7845568222,False,...,True,False,False,False,True,False,False,False,False,False
2,2,False,30,5,0,1,5,False,1183902.7845568222,False,...,True,False,False,False,True,False,False,False,False,False
3,2,True,13,7,0,1,5,False,1183902.7845568222,False,...,True,False,False,False,True,False,False,False,False,False
4,1,False,23,3,0,1,7,False,3262.780625864375,False,...,True,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,2,False,45,4,0,1,4,False,605870.8697948474,False,...,True,False,False,False,True,False,False,False,False,False
666,3,False,77,2,0,1,5,False,33858.13232104739,False,...,True,False,False,False,True,False,False,False,False,False
667,2,False,42,6,0,1,4,False,266336.5801759236,False,...,True,False,False,False,True,False,False,False,False,False
668,1,False,38,4,0,1,4,False,560856.5575155583,False,...,True,False,False,False,True,False,False,False,False,False


In [175]:
# Fix all the data types

df.dtypes

COLLISION_SEVERITY       category
AT_FAULT                     bool
PARTY_AGE                  object
DAY_OF_WEEK                object
PARTY_NUMBER_KILLED        object
PARTY_NUMBER_INJURED       object
road_class                 object
island_id                    bool
volume                     object
signal_id                    bool
PARTY_SEX_F                  bool
PARTY_SEX_M                  bool
PARTY_SEX_X                  bool
RACE_B                       bool
RACE_H                       bool
RACE_O                       bool
RACE_W                       bool
LIGHTING_A                   bool
LIGHTING_B                   bool
LIGHTING_C                   bool
LIGHTING_D                   bool
LIGHTING_E                   bool
WEATHER_1_A                  bool
WEATHER_1_B                  bool
WEATHER_1_C                  bool
WEATHER_1_E                  bool
WEATHER_1_F                  bool
PRIMARY_COLL_FACTOR_A        bool
PRIMARY_COLL_FACTOR_B        bool
PRIMARY_COLL_F

In [180]:
df['volume']

0      188606.62996098422
1      1183902.7845568222
2      1183902.7845568222
3      1183902.7845568222
4       3262.780625864375
              ...        
665     605870.8697948474
666     33858.13232104739
667     266336.5801759236
668     560856.5575155583
669     8842.145632099371
Name: volume, Length: 670, dtype: object

In [183]:
cols = ['PARTY_AGE', 'DAY_OF_WEEK', 'PARTY_NUMBER_KILLED', 'PARTY_NUMBER_INJURED', 'road_class']
df[cols] = df[cols].astype(int)

df['volume'] = df['volume'].astype(float)
df['volume'] = df['volume'].fillna(0).astype(int)


df.dtypes

COLLISION_SEVERITY       category
AT_FAULT                     bool
PARTY_AGE                   int64
DAY_OF_WEEK                 int64
PARTY_NUMBER_KILLED         int64
PARTY_NUMBER_INJURED        int64
road_class                  int64
island_id                    bool
volume                      int64
signal_id                    bool
PARTY_SEX_F                  bool
PARTY_SEX_M                  bool
PARTY_SEX_X                  bool
RACE_B                       bool
RACE_H                       bool
RACE_O                       bool
RACE_W                       bool
LIGHTING_A                   bool
LIGHTING_B                   bool
LIGHTING_C                   bool
LIGHTING_D                   bool
LIGHTING_E                   bool
WEATHER_1_A                  bool
WEATHER_1_B                  bool
WEATHER_1_C                  bool
WEATHER_1_E                  bool
WEATHER_1_F                  bool
PRIMARY_COLL_FACTOR_A        bool
PRIMARY_COLL_FACTOR_B        bool
PRIMARY_COLL_F

In [184]:
# Define independent variables (all columns except the target)
independent_vars = df.columns.difference(["COLLISION_SEVERITY"])
independent_vars

Index(['AT_FAULT', 'DAY_OF_WEEK', 'LIGHTING_A', 'LIGHTING_B', 'LIGHTING_C',
       'LIGHTING_D', 'LIGHTING_E', 'PARTY_AGE', 'PARTY_NUMBER_INJURED',
       'PARTY_NUMBER_KILLED', 'PARTY_SEX_F', 'PARTY_SEX_M', 'PARTY_SEX_X',
       'PED_ACTION_B', 'PED_ACTION_C', 'PED_ACTION_D', 'PED_ACTION_E',
       'PED_ACTION_F', 'PED_ACTION_G', 'PRIMARY_COLL_FACTOR_A',
       'PRIMARY_COLL_FACTOR_B', 'PRIMARY_COLL_FACTOR_C',
       'PRIMARY_COLL_FACTOR_D', 'RACE_B', 'RACE_H', 'RACE_O', 'RACE_W',
       'WEATHER_1_A', 'WEATHER_1_B', 'WEATHER_1_C', 'WEATHER_1_E',
       'WEATHER_1_F', 'island_id', 'road_class', 'signal_id', 'volume'],
      dtype='object')

In [185]:
# Specify and calibrate the Ordered Logit Model
model = OrderedModel(
    df["COLLISION_SEVERITY"],
    df[independent_vars],
    distr="logit"
)

result = model.fit(method="bfgs")
print(result.summary())

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [ ]:
# Compute and display Odds Ratios
odds_ratios = np.exp(result.params)
print("\nOdds Ratios:\n", odds_ratios)

# Predict probabilities for each severity level
predicted_probs = result.predict()

# Convert predictions to a DataFrame
predicted_probs_df = pd.DataFrame(predicted_probs)

# Determine the most likely severity level for each observation
df["Predicted_Severity"] = predicted_probs_df.idxmax(axis=1)

# Display performance metrics with a confusion matrix (objective results only)
confusion_matrix = pd.crosstab(df["COLLISION_SEVERITY"], df["Predicted_Severity"],
                               rownames=['Actual'], colnames=['Predicted'])
print("\nConfusion Matrix:\n", confusion_matrix)


NameError: name 'result' is not defined